1. 문서내용읽기
2. 문서쪼개기
3. 임베딩 -> 백터 데이터베이스에 저장
4. 질문이 있을때, 백터 데이터베이스에 유사도 검색
5. 유사도 검색으로 가져온 문서를 LLM에 질문과 같이 전달

In [9]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,
)

loader = Docx2txtLoader('./tax.docx')

document_list = loader.load()

split_documents = text_splitter.split_documents(document_list)

C:\Users\u1ghd\AppData\Local\Temp\ipykernel_4700\2212004060.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import Docx2txtLoader


In [ ]:
split_documents

In [20]:
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings

load_dotenv()

embedding = UpstageEmbeddings(model='solar-embedding-1-large')

In [ ]:
%pip install langchain-chroma

In [ ]:
embedding.embed_documents(["테스트 문장입니다."])

In [4]:
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [19]:
%pip install --upgrade --quiet langchain-pinecone langchain-openai langchain

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import os
from dotenv import load_dotenv

load_dotenv()

pinecone_api_key = os.getenv("PINECONE_API_KEY")

print(pinecone_api_key is not None)

True


In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

splits = text_splitter.split_documents(document_list)

print("원본 문서 수:", len(document_list))
print("분할 후 문서 수:", len(splits))

원본 문서 수: 1
분할 후 문서 수: 320


In [23]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=pinecone_api_key)

pc.create_index(
    name="tax-vector-index",
    dimension=4096,
    metric="cosine",
    spec=ServerlessSpec(
        cloud="aws",
        region="us-east-1"
    )
)

{
    "name": "tax-vector-index",
    "metric": "cosine",
    "host": "tax-vector-index-rg9fj2q.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 4096,
    "deletion_protection": "disabled",
    "tags": null
}

In [27]:
print(len(embedding.embed_query("차원 확인")))

4096


In [26]:
print(pc.list_indexes())

[{
    "name": "tax-index",
    "metric": "cosine",
    "host": "tax-index-rg9fj2q.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 1024,
    "deletion_protection": "disabled",
    "tags": null
}, {
    "name": "tax-vector-index",
    "metric": "cosine",
    "host": "tax-vector-index-rg9fj2q.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 4096,
    "deletion_protection": "disabled",
    "tags": null
}]


In [28]:
import os

from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

index_name = 'tax-vector-index'
pinecone_api_key = os.environ.get("PINECONE_API_KEY")
pc = Pinecone(api_key=pinecone_api_key)
index = pc.Index(index_name)


database = PineconeVectorStore.from_documents(split_documents, embedding, index_name=index_name)

In [ ]:
from langchain_chroma import Chroma

database = Chroma.from_documents(documents=split_documents, embedding=embedding,collection_name= 'chroma-tax', persist_directory="./chroma")

In [29]:
query = '연봉 5천만원인 직장인의 소득세는 얼마인가요?'
retrieved_docs = database.similarity_search(query)

In [31]:
index.describe_index_stats()

{'dimension': 4096,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 195}},
 'total_vector_count': 195,
 'vector_type': 'dense'}

In [30]:
retrieved_docs

[Document(id='0210edaf-a8d6-4320-af89-0ea210b14b36', metadata={'source': './tax.docx'}, page_content='2. 2명인 경우: 연 55만원\n\n3. 3명 이상인 경우: 연 55만원과 2명을 초과하는 1명당 연 40만원을 합한 금액\n\n② 삭제<2017. 12. 19.>\n\n③ 해당 과세기간에 출산하거나 입양 신고한 공제대상자녀가 있는 경우 다음 각 호의 구분에 따른 금액을 종합소득산출세액에서 공제한다.<신설 2015. 5. 13., 2016. 12. 20.>\n\n1. 출산하거나 입양 신고한 공제대상자녀가 첫째인 경우: 연 30만원\n\n2. 출산하거나 입양 신고한 공제대상자녀가 둘째인 경우: 연 50만원\n\n3. 출산하거나 입양 신고한 공제대상자녀가 셋째 이상인 경우: 연 70만원\n\n④ 제1항 및 제3항에 따른 공제를 “자녀세액공제”라 한다.<신설 2015. 5. 13., 2017. 12. 19.>\n\n[본조신설 2014. 1. 1.]\n\n[종전 제59조의2는 제59조의5로 이동 <2014. 1. 1.>]\n\n\n\n제59조의3(연금계좌세액공제) ① 종합소득이 있는 거주자가 연금계좌에 납입한 금액 중 다음 각 호에 해당하는 금액을 제외한 금액(이하 “연금계좌 납입액”이라 한다)의 100분의 12[해당 과세기간에 종합소득과세표준을 계산할 때 합산하는 종합소득금액이 4천 500만원 이하(근로소득만 있는 경우에는 총급여액 5천 500만원 이하)인 거주자에 대해서는 100분의 15]에 해당하는 금액을 해당 과세기간의 종합소득산출세액에서 공제한다. 다만, 연금계좌 중 연금저축계좌에 납입한 금액이 연 600만원을 초과하는 경우에는 그 초과하는 금액은 없는 것으로 하고, 연금저축계좌에 납입한 금액 중 600만원 이내의 금액과 퇴직연금계좌에 납입한 금액을 합한 금액이 연 900만원을 초과하는 경우에는 그 초과하는 금액은 없는 것으로 한다. <개정 2014. 12. 23., 2015.

문서를 벡터로 변환하는 임베딩 = solar-embedding-1-large
질문에 답변하는 대화형 AI = solar-mini4

In [32]:
from dotenv import load_dotenv
from langchain_upstage import ChatUpstage

load_dotenv()

llm = ChatUpstage(model="solar-mini4")

In [33]:
prompt = f"""[Identity]
- 당신은 최고의 한국 소득세 전문가 입니다
- [Context]를 참고해서 사용자의 질문에 답변해주세요

[Context]
{retrieved_docs}

Question: {query}
"""

In [34]:
ai_message = llm.invoke(prompt)

In [35]:
ai_message.content

'연봉 5천만원인 직장인의 소득세(근로소득세) 계산 흐름을 **2024~2026년 개정 세법 기준으로** 단계별로 설명드리겠습니다.  \n다만, 정확한결정세액(세후 연봉)을 알려면 **소득공제 항목(신용카드, 퇴직연금, 보험료, 의료비 등)**과 **근로소득 세액공제**까지 반영해야 해서, 여기서는 **일반적인 과세표준 구조를 기준으로 한 근사치(추정액)**를 계산하겠습니다.\n\n---\n\n## 1. 먼저 정리해야 할 개념\n\n연봉 5,000만 원이라고 해서 바로 소득세를 계산하는 것이 아니라, 다음 순서로 계산합니다.\n\n1. **총급여액** = 연봉  \n2. **근로소득공제** 적용  \n3. **근로소득금액** 계산  \n4. **종합소득공제** 적용  \n5. **과세표준** 계산  \n6. **산출세액** 계산  \n7. **세액공제·감면** 적용  \n8. **결정세액** 계산  \n9. **기납부세액(원천징수)**과의 차이로 **추가 납부 또는 환급** 결정\n\n---\n\n## 2. 연봉 5,000만 원에 대한 근로소득공제\n\n 근로소득공제는 다음과 같습니다.\n\n| 총급여액 구간 | 공제액 |\n|---|---:|\n| 500만 원 이하 | 총급여액의 70% |\n| 500만 원 초과 1,500만 원 이하 | 350만 원 + 500만 원 초과분의 40% |\n| 1,500만 원 초과 4,500만 원 이하 | 750만 원 + 1,500만 원 초과분의 15% |\n| 4,500만 원 초과 1억 원 이하 | 1,200만 원 + 4,500만 원 초과분의 5% |\n| 1억 원 초과 | 1,650만 원 + 1억 원 초과분의 2% |\n\n연봉 5,000만 원은 **4,500만 원 초과 1억 원 이하** 구간입니다.\n\n계산하면,\n\n\\[\n1,200만 원 + (5,000만 원 - 4,500만 원) \\times 5\\%\n\\]\n\n\\[\n= 1,200만 원 + 500만 원 \\times 5\\%\n\\]\n\n\\[\n= 1,2

In [1]:
from langchain_classic import hub

prompt = hub.pull(
    "rlm/rag-prompt",
    dangerously_pull_public_prompt=True
)

C:\Users\u1ghd\AppData\Local\Temp\ipykernel_3580\3115062803.py:3: LangChainDeprecationWarning: langchain_classic.hub.pull is deprecated. Use the LangSmith SDK instead.
  prompt = hub.pull(


TypeError: pull() got an unexpected keyword argument 'dangerously_pull_public_prompt'